# ROGII — Wellbore Geology Prediction
## Notebook 1: Exploratory Data Analysis (EDA)

This notebook explores the training and test data to understand:
- Data structure and dimensions
- Distribution of key features (GR, TVT, trajectory)
- Well-level variability
- Missing values and outliers
- Relationships between features and the target

## 0. Setup

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
import seaborn as sns
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

plt.rcParams['figure.figsize'] = (12, 5)
plt.rcParams['axes.grid'] = True
plt.rcParams['grid.alpha'] = 0.3
sns.set_palette('husl')

DATA_DIR = Path('../data')
print('Data directory:', DATA_DIR.resolve())

## 1. Load Data

In [ ]:
# Load training data
train = pd.read_csv(DATA_DIR / 'train.csv')
test  = pd.read_csv(DATA_DIR / 'test.csv')
sample_sub = pd.read_csv(DATA_DIR / 'sample_submission.csv')

print(f'Train shape : {train.shape}')
print(f'Test shape  : {test.shape}')
print(f'Sample sub  : {sample_sub.shape}')

In [ ]:
train.head(10)

In [ ]:
test.head(10)

In [ ]:
train.dtypes

## 2. Missing Values

In [ ]:
def missing_report(df, label=''):
    miss = df.isnull().sum()
    pct  = miss / len(df) * 100
    report = pd.DataFrame({'missing': miss, 'pct': pct}).query('missing > 0').sort_values('pct', ascending=False)
    print(f'\n--- {label} ---')
    if report.empty:
        print('No missing values!')
    else:
        print(report.to_string())
    return report

train_miss = missing_report(train, 'TRAIN')
test_miss  = missing_report(test,  'TEST')

## 3. Target Distribution (TVT)

In [ ]:
target_col = 'TVT'  # adjust if the actual column name differs

fig, axes = plt.subplots(1, 3, figsize=(18, 4))

axes[0].hist(train[target_col].dropna(), bins=100, color='steelblue', edgecolor='none')
axes[0].set_title('TVT Distribution')
axes[0].set_xlabel('TVT (m)')

axes[1].boxplot(train[target_col].dropna(), vert=True, patch_artist=True,
                boxprops=dict(facecolor='lightblue'))
axes[1].set_title('TVT Boxplot')
axes[1].set_ylabel('TVT (m)')

# QQ-style: sorted values
sorted_tvt = np.sort(train[target_col].dropna())
axes[2].plot(sorted_tvt, np.linspace(0, 1, len(sorted_tvt)), lw=2)
axes[2].set_title('TVT Empirical CDF')
axes[2].set_xlabel('TVT (m)')
axes[2].set_ylabel('Cumulative probability')

plt.tight_layout()
plt.show()

print(train[target_col].describe().round(2))

## 4. Gamma-Ray Log Distribution

In [ ]:
gr_col = 'GR'  # adjust if needed

fig, axes = plt.subplots(1, 2, figsize=(14, 4))

axes[0].hist(train[gr_col].dropna(), bins=100, color='darkorange', edgecolor='none', alpha=0.7, label='Train')
axes[0].hist(test[gr_col].dropna(),  bins=100, color='steelblue',  edgecolor='none', alpha=0.7, label='Test')
axes[0].axvline(50,  color='green', ls='--', lw=1.5, label='GR=50 (reservoir threshold)')
axes[0].axvline(75,  color='red',   ls='--', lw=1.5, label='GR=75 (shale threshold)')
axes[0].set_title('GR Distribution: Train vs Test')
axes[0].set_xlabel('GR (API units)')
axes[0].legend()

axes[1].scatter(train[gr_col].sample(5000, random_state=42),
                train[target_col].sample(5000, random_state=42),
                alpha=0.3, s=5, c='steelblue')
axes[1].set_title('GR vs TVT')
axes[1].set_xlabel('GR (API units)')
axes[1].set_ylabel('TVT (m)')

plt.tight_layout()
plt.show()

## 5. Well-Level Analysis

In [ ]:
well_col = 'well_id'  # adjust if needed

n_wells_train = train[well_col].nunique()
n_wells_test  = test[well_col].nunique()
print(f'Unique wells in train: {n_wells_train}')
print(f'Unique wells in test : {n_wells_test}')

well_stats = train.groupby(well_col).agg(
    n_samples=(target_col, 'count'),
    tvt_mean=(target_col, 'mean'),
    tvt_std=(target_col, 'std'),
    gr_mean=(gr_col, 'mean'),
    gr_std=(gr_col, 'std')
).reset_index()

well_stats.describe().round(2)

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 4))

axes[0].hist(well_stats['n_samples'], bins=30, color='steelblue', edgecolor='white')
axes[0].set_title('Samples per Well')
axes[0].set_xlabel('# samples')

axes[1].hist(well_stats['tvt_mean'], bins=30, color='darkorange', edgecolor='white')
axes[1].set_title('Mean TVT per Well')
axes[1].set_xlabel('Mean TVT (m)')

axes[2].hist(well_stats['tvt_std'], bins=30, color='green', edgecolor='white')
axes[2].set_title('TVT Std per Well (intra-well variability)')
axes[2].set_xlabel('Std TVT (m)')

plt.tight_layout()
plt.show()

## 6. TVT Profile Along a Single Well

In [ ]:
depth_col = 'DEPT'  # measured depth column — adjust if needed

# Pick the first 3 wells to visualize
sample_wells = train[well_col].unique()[:3]

fig, axes = plt.subplots(len(sample_wells), 2, figsize=(16, 5 * len(sample_wells)))

for i, w in enumerate(sample_wells):
    well_df = train[train[well_col] == w].sort_values(depth_col)
    
    axes[i, 0].plot(well_df[depth_col], well_df[gr_col], color='darkorange', lw=0.8)
    axes[i, 0].set_title(f'Well {w} — GR Log')
    axes[i, 0].set_xlabel('Measured Depth (m)')
    axes[i, 0].set_ylabel('GR (API)')
    axes[i, 0].axhline(50, color='green', ls='--', lw=1, label='GR=50')
    axes[i, 0].axhline(75, color='red',   ls='--', lw=1, label='GR=75')
    axes[i, 0].legend(fontsize=8)
    
    axes[i, 1].plot(well_df[depth_col], well_df[target_col], color='steelblue', lw=1.2)
    axes[i, 1].set_title(f'Well {w} — TVT (Target)')
    axes[i, 1].set_xlabel('Measured Depth (m)')
    axes[i, 1].set_ylabel('TVT (m)')

plt.tight_layout()
plt.show()

## 7. Wellbore Trajectory in 3D

In [ ]:
from mpl_toolkits.mplot3d import Axes3D

sample_well = train[well_col].unique()[0]
well_df = train[train[well_col] == sample_well].sort_values(depth_col)

fig = plt.figure(figsize=(12, 8))
ax = fig.add_subplot(111, projection='3d')

sc = ax.scatter(
    well_df['X'], well_df['Y'], well_df['Z'],
    c=well_df[target_col], cmap='RdYlGn_r', s=5, alpha=0.7
)
plt.colorbar(sc, ax=ax, label='TVT (m)', shrink=0.5)
ax.set_title(f'3D Trajectory — Well {sample_well}\n(color = TVT)')
ax.set_xlabel('X (m)')
ax.set_ylabel('Y (m)')
ax.set_zlabel('Z (m)')
plt.tight_layout()
plt.show()

## 8. Feature Correlation Matrix

In [ ]:
numeric_cols = train.select_dtypes(include=[np.number]).columns.tolist()

corr = train[numeric_cols].corr()

# Sort columns by correlation with target
if target_col in corr.columns:
    sort_order = corr[target_col].abs().sort_values(ascending=False).index
    corr = corr.loc[sort_order, sort_order]

plt.figure(figsize=(14, 11))
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, mask=mask, cmap='RdBu_r', center=0, annot=True, fmt='.2f',
            linewidths=0.5, square=True, cbar_kws={'shrink': 0.7})
plt.title('Feature Correlation Matrix (sorted by |corr with TVT|)')
plt.tight_layout()
plt.show()

## 9. GR Normalization Analysis

GR logs can drift significantly between wells due to borehole conditions and tool calibration. Normalization is often critical.

In [ ]:
# Per-well GR statistics
gr_stats = train.groupby(well_col)[gr_col].agg(['mean', 'std', 'min', 'max'])

fig, axes = plt.subplots(1, 2, figsize=(14, 4))

axes[0].scatter(gr_stats['mean'], gr_stats['std'], alpha=0.5, s=20)
axes[0].set_title('Per-Well GR: Mean vs Std')
axes[0].set_xlabel('Mean GR (API)')
axes[0].set_ylabel('Std GR (API)')

# GR range per well
gr_range = gr_stats['max'] - gr_stats['min']
axes[1].hist(gr_range, bins=40, color='purple', edgecolor='white')
axes[1].set_title('Per-Well GR Range (max - min)')
axes[1].set_xlabel('GR Range (API units)')

plt.tight_layout()
plt.show()

print(f'GR mean ranges from {gr_stats["mean"].min():.1f} to {gr_stats["mean"].max():.1f} API across wells')
print('→ Per-well normalization recommended!')

## 10. Lag / Auto-Correlation of TVT

TVT evolves smoothly along depth — understanding its autocorrelation helps choose optimal lag features.

In [ ]:
from statsmodels.graphics.tsaplots import plot_acf

# Use the largest well for stable autocorrelation estimate
largest_well = well_stats.sort_values('n_samples', ascending=False).iloc[0][well_col]
well_series = train[train[well_col] == largest_well].sort_values(depth_col)[target_col].dropna()

fig, axes = plt.subplots(1, 2, figsize=(16, 4))

# Manual autocorrelation
lags = range(1, 51)
acf_vals = [well_series.autocorr(lag=k) for k in lags]
axes[0].bar(list(lags), acf_vals, color='steelblue')
axes[0].axhline(0, color='black', lw=0.8)
axes[0].axhline(0.05, color='red', ls='--', lw=1)
axes[0].axhline(-0.05, color='red', ls='--', lw=1)
axes[0].set_title(f'TVT Autocorrelation — Well {largest_well}')
axes[0].set_xlabel('Lag (# depth samples)')
axes[0].set_ylabel('Autocorrelation')

# GR autocorrelation
gr_series = train[train[well_col] == largest_well].sort_values(depth_col)[gr_col].dropna()
gr_acf = [gr_series.autocorr(lag=k) for k in lags]
axes[1].bar(list(lags), gr_acf, color='darkorange')
axes[1].axhline(0, color='black', lw=0.8)
axes[1].set_title(f'GR Autocorrelation — Well {largest_well}')
axes[1].set_xlabel('Lag (# depth samples)')
axes[1].set_ylabel('Autocorrelation')

plt.tight_layout()
plt.show()

## 11. Train vs Test Distribution Shift

In [ ]:
# Check for distribution shift in numeric features
common_cols = [c for c in train.select_dtypes(include=[np.number]).columns
               if c in test.columns and c != target_col]

n_cols = len(common_cols)
n_rows = (n_cols + 2) // 3

fig, axes = plt.subplots(n_rows, 3, figsize=(18, 4 * n_rows))
axes = axes.flatten()

for i, col in enumerate(common_cols):
    lo = min(train[col].quantile(0.01), test[col].quantile(0.01))
    hi = max(train[col].quantile(0.99), test[col].quantile(0.99))
    bins = np.linspace(lo, hi, 50)
    
    axes[i].hist(train[col].dropna(), bins=bins, density=True,
                 alpha=0.6, color='steelblue', label='Train')
    axes[i].hist(test[col].dropna(),  bins=bins, density=True,
                 alpha=0.6, color='darkorange', label='Test')
    axes[i].set_title(col)
    axes[i].legend(fontsize=7)

for j in range(i + 1, len(axes)):
    axes[j].set_visible(False)

plt.suptitle('Train vs Test Feature Distributions', y=1.01, fontsize=14)
plt.tight_layout()
plt.show()

## 12. EDA Summary

| Finding | Implication |
|---|---|
| GR distribution varies significantly across wells | Normalize GR per well or against reference well |
| TVT has strong autocorrelation up to ~20 lags | Include lag-k features (k=1..20) |
| TVT distribution may be bimodal or skewed | Consider log-transform or quantile transform |
| Some wells have very few samples | May need to exclude tiny wells or handle separately |
| Train and test GR ranges overlap well | No severe covariate shift expected |
| 3D trajectory shows clear depth/spatial structure | X, Y, Z and derived tortuosity are important features |

### Next steps
→ Proceed to `02_modeling.ipynb` for feature engineering and model training.